# MahaMonsoon-X — Load, preprocess, and export ERA5 to CSV

This notebook loads the ERA5 monthly GRIB variables, converts the grid into one tabular DataFrame, removes columns with **more than 50% missing values**, imputes remaining missing numeric values with their column median, standardizes weather variables, and saves the complete processed DataFrame as CSV.

**Output:** `data/processed/era5_processed.csv`

Coordinates (`timestamp`, `latitude`, `longitude`) are preserved and are not standardized. The original GRIB file is not modified.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import gc
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import cfgrib
from IPython.display import display

RAW_PATH = Path("../../data/era5/era5_monthly_1940_2026.grib")
OUT_PATH = Path("../../data/processed/era5_processed.csv")

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {RAW_PATH.resolve()}. Run this notebook from the project root."
    )

print(f"Input file: {RAW_PATH.resolve()}")
print(f"GRIB file size: {RAW_PATH.stat().st_size / (1024**2):,.2f} MiB")

## 2. Load the complete dataset into one DataFrame

In [ ]:
# Read only GRIB headers first to discover the variable names. This does not decode
# the full grid values.
from eccodes import codes_grib_new_from_file, codes_get, codes_release

short_names = []
message_count = 0

with RAW_PATH.open("rb") as grib_file:
    while True:
        gid = codes_grib_new_from_file(grib_file)
        if gid is None:
            break
        message_count += 1
        try:
            short_name = codes_get(gid, "shortName")
            if short_name not in short_names:
                short_names.append(short_name)
        finally:
            codes_release(gid)

print("GRIB messages:", f"{message_count:,}")
print("Unique GRIB variables:", len(short_names))
print("Variables:", short_names)

In [ ]:
# Decode one variable at a time to avoid cfgrib trying to discover every possible
# group at once. Each variable is converted to rows keyed by timestamp and grid point.
KEYS = ["timestamp", "latitude", "longitude"]
data = None

for number, short_name in enumerate(short_names, start=1):
    print(f"[{number}/{len(short_names)}] Loading {short_name} ...")

    try:
        ds = xr.open_dataset(
            RAW_PATH,
            engine="cfgrib",
            backend_kwargs={
                "filter_by_keys": {"shortName": short_name},
                "indexpath": ""
            }
        )
    except Exception as exc:
        warnings.warn(f"Skipping {short_name}: could not open it ({exc})")
        continue

    try:
        data_vars = list(ds.data_vars)
        if not data_vars:
            warnings.warn(f"Skipping {short_name}: no data variable found.")
            continue

        # Usually the variable name matches shortName, but use the actual name
        # returned by xarray to avoid assuming that it does.
        var_name = data_vars[0]
        da = ds[var_name]

        # Use valid_time as the row timestamp when it is a one-dimensional
        # coordinate aligned with time; otherwise retain time.
        if "valid_time" in ds.coords and ds["valid_time"].ndim == 1:
            time_dim = ds["valid_time"].dims[0]
            if time_dim in da.dims:
                da = da.assign_coords({time_dim: ds["valid_time"]})
                da = da.rename({time_dim: "timestamp"})
        elif "time" in da.dims:
            da = da.rename({"time": "timestamp"})

        frame = da.to_dataframe(name=short_name).reset_index()

        # Normalize coordinate column names.
        if "valid_time" in frame.columns and "timestamp" not in frame.columns:
            frame = frame.rename(columns={"valid_time": "timestamp"})
        elif "time" in frame.columns and "timestamp" not in frame.columns:
            frame = frame.rename(columns={"time": "timestamp"})

        # Remove duplicate/auxiliary time coordinate columns.
        keep = [c for c in KEYS if c in frame.columns] + [short_name]
        frame = frame.loc[:, list(dict.fromkeys(keep))]

        if not all(key in frame.columns for key in KEYS):
            warnings.warn(
                f"Skipping {short_name}: expected coordinates {KEYS}, "
                f"found {frame.columns.tolist()}"
            )
            continue

        frame = frame.drop_duplicates(subset=KEYS)
        if data is None:
            data = frame
        else:
            data = data.merge(frame, on=KEYS, how="outer", sort=False)
            del frame
            gc.collect()
        print(f"    rows in combined DataFrame: {len(data):,}; columns: {len(data.columns):,}")
    finally:
        ds.close()
        del ds
        gc.collect()

if data is None:
    raise RuntimeError(
        "No variables could be loaded. Check that cfgrib and ecCodes are installed "
        "and that the GRIB file is valid."
    )

gc.collect()

data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
data = data.sort_values(KEYS, kind="stable").reset_index(drop=True)

print("\nComplete DataFrame loaded.")
print("Rows:", f"{data.shape[0]:,}")
print("Columns:", f"{data.shape[1]:,}")
print("Memory usage:", f"{data.memory_usage(deep=True).sum() / (1024**3):.2f} GiB")
display(data.head())

## 3. Handle GRIB missing-value sentinels and remove columns over 50% missing

In [ ]:
# ERA5 GRIB metadata can use 9999 as a missing-value sentinel. Replace it with NaN
# before calculating missingness. Also treat infinite numeric values as missing.
data = data.replace([9999, -9999, np.inf, -np.inf], np.nan)

feature_columns = [c for c in data.columns if c not in KEYS]
missing_fraction = data[feature_columns].isna().mean()
missing_report = (
    pd.DataFrame({
        "missing_count": data[feature_columns].isna().sum(),
        "missing_percent": missing_fraction * 100
    })
    .sort_values("missing_percent", ascending=False)
)

print("Missing values before dropping columns:")
display(missing_report)

columns_to_drop = missing_fraction[missing_fraction > 0.50].index.tolist()
print("Columns to drop (>50% missing):", columns_to_drop)

data = data.drop(columns=columns_to_drop)
feature_columns = [c for c in data.columns if c not in KEYS]

print("Shape after dropping high-missingness columns:", data.shape)

## 4. Impute remaining missing values

In [ ]:
# Impute each remaining weather variable with its median. This avoids replacing
# missing values with zero, which would introduce artificial observations.
for column in feature_columns:
    data[column] = pd.to_numeric(data[column], errors="coerce")
    median_value = data[column].median()
    if pd.isna(median_value):
        # A column that is entirely missing after numeric conversion has no usable
        # median, so remove it instead of inventing values.
        data = data.drop(columns=[column])
        print(f"Dropped {column}: no valid numeric values remain.")
    else:
        data[column] = data[column].fillna(median_value)

feature_columns = [c for c in data.columns if c not in KEYS]
print("Shape after imputation:", data.shape)
print("Remaining missing values:", int(data.isna().sum().sum()))

## 5. Standardize weather variables

In [ ]:
# Standardize only weather variables. Keep timestamp and geographic coordinates
# unchanged so that they remain meaningful for later spatial/temporal analysis.
# Constant columns are set to 0 after standardization.
scaling_stats = {}

for column in feature_columns:
    mean_value = data[column].mean()
    std_value = data[column].std(ddof=0)
    scaling_stats[column] = {"mean": float(mean_value), "std": float(std_value)}

    if pd.isna(std_value) or np.isclose(std_value, 0.0):
        data[column] = 0.0
    else:
        data[column] = (data[column] - mean_value) / std_value

print("Standardized weather columns:", len(feature_columns))
display(data[feature_columns].describe().T.head(20))

## 6. Save the complete processed DataFrame to CSV

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save in chunks to avoid creating a second full-size string representation in RAM.
# Write the header once, then append subsequent chunks.
if OUT_PATH.exists():
    OUT_PATH.unlink()

chunk_size = 100_000
for start in range(0, len(data), chunk_size):
    chunk = data.iloc[start:start + chunk_size]
    chunk.to_csv(
        OUT_PATH,
        mode="w" if start == 0 else "a",
        header=(start == 0),
        index=False,
        date_format="%Y-%m-%dT%H:%M:%S"
    )
    print(f"Saved rows: {min(start + len(chunk), len(data)):,}/{len(data):,}")

print("\nCSV saved:", OUT_PATH.resolve())
print("CSV size:", f"{OUT_PATH.stat().st_size / (1024**3):.2f} GiB")
print("Final shape:", data.shape)
print("Final columns:", data.columns.tolist())
print("Missing values in final DataFrame:", int(data.isna().sum().sum()))

## 7. Verify the saved CSV

In [ ]:
# Read a small sample only, rather than reloading the entire CSV into memory.
saved_sample = pd.read_csv(OUT_PATH, nrows=5)
print("Saved CSV sample shape:", saved_sample.shape)
print("Saved CSV columns:", saved_sample.columns.tolist())
display(saved_sample)

print("\nProcessing complete. Use data/processed/era5_processed.csv for subsequent work.")